# 06 Y1打分：没省钱／不划算／多花
旧版route3：上下文规则弱标签 → 字符TF-IDF → L1 Logistic。完整建模池22,831行，不是仅有X的交集。分数是对弱标签的预测，不是人工标注准确率或实际行为概率。

默认读取旧模型后，从原文重新变换与预测；这一步不是重新拟合。要完整重新训练，请在第4步显式开启RUN_TRAINING。

首次使用：在python目录的终端执行 python -m pip install -r requirements.txt，然后执行 jupyter lab。

In [1]:
from pathlib import Path
import sys
# 从python目录或项目根启动均可；没有固定机器路径
PYTHON_DIR = Path.cwd() if (Path.cwd() / 'analysis_helpers.py').exists() else Path.cwd() / 'python'
if not (PYTHON_DIR / 'analysis_helpers.py').exists():
    raise FileNotFoundError('请从lasso论文或lasso论文/python目录打开此Notebook')
sys.path.insert(0, str(PYTHON_DIR.resolve()))
try:
    import analysis_helpers as h
except ImportError as exc:
    raise ImportError("缺少依赖，请在python目录运行：python -m pip install -r requirements.txt；安装后重启内核") from exc
print(h.runtime_check())
NAME = 'Y1'

{'python': '3.12.14', 'numpy': '2.3.5', 'scipy': '1.17.0', 'scikit-learn': '1.8.0'}


## 1 读取完整建模池与固定分组
原60%训练／40%测试和train内5折全部保留。近重复family不跨侧；Topic可跨侧。分层变量是X档×弱标签；H/A标记仅作来源诊断，不用于分层或预测特征。

In [2]:
df, plan = h.load_target(NAME)
print('全池:', len(df), 'train:', len(plan['train_ids']), 'test:', len(plan['test_ids']))
print('seed:', plan['outer_seed'], 'inner seed:', plan['inner_seed'])
print('弱标签正例:', int(df.weak_label.sum()))
display(df[['row_id','text','weak_label','split','cv_validation_fold']].head(5))

全池: 22831 train: 13698 test: 9133
seed: 20264019 inner seed: 20364019
弱标签正例: 225


,row_id,text,weak_label,split,cv_validation_fold
0,BILI:BV11ZGX64EDa:checkpoint089_capture_1,很久没回来了，居然有这么多评论，有这么多兄弟在这个平台遇到各种糟心事儿[笑哭]只能说远离抖音...,0,train,1
1,BILI:BV11ZGX64EDa:checkpoint089_capture_10,抖音买手机靠谱吗？我看别人说这些抖音自营其实也是外包给小商家的，手机有问题我直接去官方售后维...,0,test,
2,BILI:BV11ZGX64EDa:checkpoint089_capture_11,手机只要是全新未激活的就没问题，如果出问题找品牌售后就行了。,0,test,
3,BILI:BV11ZGX64EDa:checkpoint089_capture_12,我在抖音买过两次，大的官方店的东西没问题那就真的没问题就正常购物，但凡买的东西有问题那就没人...,0,train,3
4,BILI:BV11ZGX64EDa:checkpoint089_capture_13,其实不同的电商，虽然都宣称官方旗舰店，也有可能是不同的商家运营的。所以一个发顺丰，一个发圆通...,0,test,


## 2 逐条重新执行冻结弱标签规则
词典与上下文否定／条件／说话对象判断在y_labeler.py和弱标签词典.json中。规则结果须与冻结标签一致才继续，不将规则命中当人工金标准。

In [3]:
label_check = h.check_weak_labels(NAME, df)
print(label_check)

{'N': 22831, 'positive_N': 225, 'mismatch_N': 0}


## 3 确认训练方法
每一折的词表和IDF只在该折fit侧生成；字符2–3 gram，min_df=2，L2归一。目标函数是平均logistic loss＋λ×L1，截距不惩罚，KKT误差≤1e-7。只用train内5折加权logloss选择λ1se；测试集不参与选择。

In [4]:
import json
selection = json.loads((h.target_folder(NAME) / '旧版模型' / '模型选择.json').read_text(encoding='utf-8'))
print('旧版 λ1se:', selection['lambda_1se'])
print('λ网格:', selection['grid'])

旧版 λ1se: 9.999999999999999e-06
λ网格: [1e-06, 3.162277660168379e-06, 9.999999999999999e-06, 3.1622776601683795e-05, 0.0001, 0.00031622776601683794, 0.001, 0.0031622776601683794, 0.01, 0.03162277660168379, 0.1]


## 4 可选：真正完整重新训练
设为True会实际执行5折CV、λ选择和完整训练路径，通常需要数分钟，慢机器可能更久。输出写入本目标数据目录下新建的重跑模型子目录，旧模型不被覆盖。数值版本不同可能有微小差异。

In [5]:
RUN_TRAINING = False
if RUN_TRAINING:
    training_result = h.train_target(NAME)
    print(training_result)
else:
    print('本次未重新训练；下一步使用已验证旧模型从原文重新评分')

本次未重新训练；下一步使用已验证旧模型从原文重新评分


## 5 从测试侧原文重新计算预测分数
全部9,133条测试数据都会评分。只在08关联分析中再取有X的交集，不按正标签筛选。重新计算结果逐行与旧评分对照。

In [6]:
scores = h.score_target(NAME, df, plan)
print('评分行数:', len(scores), '与旧评分最大绝对差:', scores.attrs['max_abs_error_vs_old'])
display(scores[['row_id','weak_label','score']].head(10))

评分行数: 9133 与旧评分最大绝对差: 1.1102230246251565e-16


,row_id,weak_label,score
0,BILI:BV11ZGX64EDa:checkpoint089_capture_10,0,0.001264
1,BILI:BV11ZGX64EDa:checkpoint089_capture_11,0,0.001194
2,BILI:BV11ZGX64EDa:checkpoint089_capture_13,0,0.001192
3,BILI:BV11ZGX64EDa:checkpoint089_capture_14,0,0.000796
4,BILI:BV11ZGX64EDa:checkpoint089_capture_15,0,0.002769
5,BILI:BV11ZGX64EDa:checkpoint089_capture_18,0,0.000796
6,BILI:BV11ZGX64EDa:checkpoint089_capture_20,0,0.000905
7,BILI:BV11ZGX64EDa:checkpoint089_capture_21,0,0.002344
8,BILI:BV11ZGX64EDa:checkpoint089_capture_24,0,0.000267
9,BILI:BV11ZGX64EDa:checkpoint089_capture_28,0,0.002104


## 6 可选保存重算评分

In [7]:
WRITE_OUTPUT = False
if WRITE_OUTPUT:
    output = h.target_folder(NAME) / '重新计算测试集评分.csv'
    scores.to_csv(output, index=False, encoding='utf-8-sig')
    print(output.relative_to(h.ROOT))